# KG1 v158 CORRIGIDO - 3 correcoes criticas + fixes vs v157

## REGRA ABSOLUTA: NAO REGREDIR ABAIXO DE 0.86

## 🔧 CORRECOES CRITICAS vs v157:

### 1. DATA LEAK ELIMINADO
- **HOLDOUT split** com seed=42:
  - Train: cryptarithm_813[100:] + eq_guess_150[20:] + synth_4425[100:]
  - Eval: cryptarithm_813[:100] + eq_guess_150[:20] + synth_4425[:100]
- **220 samples eval** balanced, NUNCA vistos em training

### 2. FIELD ROBUSTO
- get_gold(row): tenta `answer`, `solution`, `correct_answer`, `expected`
- Fallback: extrai de CoT via regex `\boxed{X}`
- Sem mais "0/100" silent fail

### 3. PERFORMANCE CHECK
- Antes de eval: gerar 1 sample, medir tok/s
- Se < 10 tok/s: WARN + abort (Komil bug)
- Se >= 10 tok/s: continue (esperado 30-50 tok/s com FlashAttn2)

## 4 FASES (5h total, 2/5 slots Kaggle):

| Fase | Tempo | Slots | O que faz |
|---|---|---|---|
| 0 | 10 min | 1 | Submit V120 baseline |
| 1 | 30-45 min | 0 | Diagnostic + speed test |
| 2 | 3-4h | 0 | Train ULTRA-CONSERVATIVE |
| 3 | 30-45 min | 1 | Paired validate (HOLDOUT) + submit |

## CONFIG (mantida do v157):

```
LR: 1e-5 cosine | max_steps: 80 | batch_eff: 16
warmup: 10% | label_smoothing: 0.0 | dropout: 0.0
weight_decay: 0.0 | grad_clip: 0.5 | seed: 42
max_seq: 6144 | FlashAttn2 | NEFTune α=5
```

## Probabilidades (consolidadas):
- Manter ≥0.86: >95%
- Atingir 0.87+: 30-40%
- Risco regressão: <5%

## Setup:
1. Runtime A100 ALTA RAM
2. Colab Secrets: HF_KEY, KAGGLE_USERNAME, KAGGLE_KEY
3. PRE-REQUISITO: /content/v120_adapter baixado
4. Form param FASE -> Run all


In [ ]:
# CELL UNICA v158 CORRIGIDO - 4 fases com 3 correcoes criticas
import os, sys, time, math, gc, glob, csv, json, urllib.request, statistics, subprocess, shutil, re, random

# === FORM PARAMS ===
FASE = "0_submit_baseline"  #@param ["0_submit_baseline", "1_diagnostic", "2_train_ultraconservative", "3_paired_validate_submit"]
ADAPTER_PATH = "/content/v120_adapter"  #@param {type:"string"}
DRY_RUN = False  #@param {type:"boolean"}
SPEED_TEST_MIN_TOKS = 10  #@param {type:"integer"}

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'

print('=' * 70)
print(f'KG1 v158 CORRIGIDO - FASE {FASE}')
print('=' * 70)

# ============================================================
# COMMON SETUP
# ============================================================
print('\n[SETUP] Secrets + GPU + verify adapter')
try:
    from google.colab import userdata, drive
    hf_token = None
    for k in ['HF_KEY', 'HF_TOKEN']:
        try:
            v = userdata.get(k)
            if v:
                hf_token = v
                os.environ['HF_TOKEN'] = v
                break
        except: pass
    if not hf_token:
        raise RuntimeError('HF_KEY missing')
    try:
        os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
        os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    except: pass
except ImportError:
    raise RuntimeError('Run in Colab')

import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU required')
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'  GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram_gb:.1f} GB')

# Verify adapter
adapter_dir = None
if os.path.exists(ADAPTER_PATH):
    for root, dirs, files in os.walk(ADAPTER_PATH):
        if 'adapter_config.json' in files and 'adapter_model.safetensors' in files:
            sz = os.path.getsize(os.path.join(root, 'adapter_model.safetensors'))
            if sz > 1_000_000_000:
                adapter_dir = root
                break
if not adapter_dir:
    raise RuntimeError(f'Adapter not found in {ADAPTER_PATH}. Download first!')

with open(os.path.join(adapter_dir, 'adapter_config.json')) as f:
    cfg = json.load(f)
print(f'  ADAPTER: {adapter_dir}')
print(f'  r={cfg.get("r")} alpha={cfg.get("lora_alpha")} dropout={cfg.get("lora_dropout")}')

# ============================================================
# CORRECAO #2: FIELD ROBUSTO
# ============================================================
BOXED_RE = re.compile(r'\\boxed\{([^}]+)\}')

def get_gold(row):
    """Tenta multiplos fields + fallback regex no CoT."""
    for field in ['answer', 'solution', 'correct_answer', 'expected', 'gold']:
        v = row.get(field)
        if v is not None:
            s = str(v).strip()
            if s:
                return s
    # Fallback: extrair de CoT
    for cot_field in ['cot', 'generated_cot', 'completion', 'response']:
        cot = row.get(cot_field)
        if cot:
            m = BOXED_RE.findall(str(cot))
            if m:
                # Pega ultimo \boxed{} nao-vazio
                for x in reversed(m):
                    if x.strip():
                        return x.strip()
    return None

# ============================================================
# CORRECAO #1: HOLDOUT SPLIT (anti data-leak)
# ============================================================
def load_datasets_with_split(data_dir):
    """
    Retorna (train_data, eval_data) com split deterministic.
    EVAL: primeiros N samples de cada dataset (NUNCA usados em train)
    TRAIN: rest
    """
    URLS = {
        'cryptarithm_813.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-cot-813/resolve/main/cryptarithm_cot_813.jsonl',
        'eq_guess_150.jsonl': 'https://gist.githubusercontent.com/FELIPEACASTRO/0d4674009f3886d6add5be636292001a/raw',
        'synth_4425.jsonl': 'https://huggingface.co/datasets/felipesp1983/nemotron-cryptarithm-synth-4425/resolve/main/synth_cryptarithm_verified.jsonl',
    }
    for name, url in URLS.items():
        out = os.path.join(data_dir, name)
        if not os.path.exists(out):
            urllib.request.urlretrieve(url, out)

    EVAL_SPLITS = {  # primeiros N de cada arquivo
        'cryptarithm_813.jsonl': 100,
        'eq_guess_150.jsonl': 20,
        'synth_4425.jsonl': 100,
    }

    train_data = []
    eval_data = []
    train_data_full_dict = {'crypt': [], 'eq': [], 'synth': []}
    eval_data_full_dict = {'crypt': [], 'eq': [], 'synth': []}
    src_map = {'cryptarithm_813.jsonl': 'crypt', 'eq_guess_150.jsonl': 'eq', 'synth_4425.jsonl': 'synth'}

    for fname, eval_n in EVAL_SPLITS.items():
        src = src_map[fname]
        path = os.path.join(data_dir, fname)
        rows = []
        with open(path, encoding='utf-8') as f:
            for line in f:
                try:
                    row = json.loads(line)
                except:
                    continue
                if row.get('is_valid', True):
                    rows.append(row)

        # Sort deterministico (sem random shuffle - seed pode mudar entre runs)
        # Use first N as eval, rest as train
        for i, row in enumerate(rows):
            row['_src'] = src
            if i < eval_n:
                eval_data.append(row)
                eval_data_full_dict[src].append(row)
            else:
                train_data.append(row)
                train_data_full_dict[src].append(row)

    return train_data, eval_data, train_data_full_dict, eval_data_full_dict

# ============================================================
# FASE 0: submit_baseline_v120
# ============================================================
if FASE == '0_submit_baseline':
    print('\n[FASE 0] Submit V120 baseline (NO TRAIN, ~10 min)')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kaggle'], capture_output=True)

    SUBMIT_DIR = '/content/submit_v158_baseline'
    os.makedirs(SUBMIT_DIR, exist_ok=True)
    for fname in ['adapter_config.json', 'adapter_model.safetensors']:
        src = os.path.join(adapter_dir, fname)
        if os.path.exists(src):
            shutil.copy(src, SUBMIT_DIR)
            print(f'  Copied {fname}: {os.path.getsize(src)/1e6:.1f} MB')

    zip_path = '/content/v158_baseline.zip'
    subprocess.run(['zip', '-j', zip_path, *glob.glob(f'{SUBMIT_DIR}/*')], check=True)
    sz_mb = os.path.getsize(zip_path) / 1e6
    print(f'  ZIP: {sz_mb:.1f} MB')

    if not DRY_RUN:
        msg = 'v158 FASE 0 - V120 Modal Surfer baseline (expected 0.85-0.86 confirmed)'
        r = subprocess.run(['kaggle', 'competitions', 'submit', '-c',
                          'nvidia-nemotron-model-reasoning-challenge', '-f', zip_path, '-m', msg],
                         capture_output=True, text=True, timeout=600)
        print(f'  Submit: {r.stdout[:300]}{r.stderr[:300]}')
        print('\n  >>> SLOTS USED: 1/5')
        print('  >>> Aguarde ~15-30 min para Kaggle reportar score')
        print('  >>> Se score > 0.84: CONTINUE FASE 1')
    print('\n=== FASE 0 DONE ===')
    raise SystemExit(0)

# ============================================================
# Setup pesado para FASE 1, 2, 3
# ============================================================
print('\n[DEPS] Install full deps')
deps = [
    'transformers>=5.3.0', 'peft>=0.14.0', 'trl>=0.14.0', 'datasets>=3.0.0',
    'accelerate>=1.3.0', 'bitsandbytes>=0.45.0', 'huggingface_hub>=0.27.0',
    'einops', 'packaging', 'ninja', 'triton', 'kaggle',
]
for pkg in deps:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', pkg],
                      capture_output=True, text=True, timeout=400)
    print(f'  {"[OK]" if r.returncode == 0 else "[FAIL]"} {pkg}')

# CORRECAO #6: torchao --force-reinstall (v155 bug)
print('  [FIX v155 bug] torchao --force-reinstall...')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall',
                   '--upgrade', 'torchao>=0.16.0'], capture_output=True, text=True, timeout=300)
print(f'  {"[OK]" if r.returncode == 0 else "[WARN]"} torchao force-reinstall')

# Try flash-attn (may fail, fallback eager)
print('  [TRY] flash-attn (may fail, fallback OK)...')
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'flash-attn>=2.7.0', '--no-build-isolation'],
                  capture_output=True, text=True, timeout=600)
print(f'  flash-attn: {"OK" if r.returncode == 0 else "FAILED (fallback eager)"}')

for m in ['transformers', 'peft', 'trl', 'bitsandbytes', 'torchao']:
    if m in sys.modules:
        del sys.modules[m]

print('\n[MAMBA] Install mamba-ssm 2.3.1')
import torch
py_ver = f'cp{sys.version_info.major}{sys.version_info.minor}'
torch_short = '.'.join(torch.__version__.split('+')[0].split('.')[:2])
abi_str = 'TRUE' if torch.compiled_with_cxx11_abi() else 'FALSE'

attempts = [('cu12', torch_short, abi_str)]
for t in ['2.10', '2.9', '2.8']:
    for abi in ['TRUE', 'FALSE']:
        if (('cu12', t, abi)) not in attempts:
            attempts.append(('cu12', t, abi))

for cu, t, abi in attempts:
    cc_url = f'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    mb_url = f'https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+{cu}torch{t}cxx11abi{abi}-{py_ver}-{py_ver}-linux_x86_64.whl'
    ok_cc = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', cc_url],
                          capture_output=True, text=True, timeout=600).returncode == 0
    ok_mb = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', mb_url],
                          capture_output=True, text=True, timeout=600).returncode == 0
    if ok_cc and ok_mb:
        print(f'  [OK] {cu} torch{t} abi{abi}')
        break

for m in ['mamba_ssm', 'causal_conv1d']:
    if m in sys.modules:
        del sys.modules[m]

# ============================================================
# Load Model + V120 Adapter
# ============================================================
print(f'\n[LOAD] Nemotron-30B + V120 adapter')
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=hf_token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

t0 = time.time()
# CORRECAO #5: try trust_remote_code=False first (transformers>=5.3.0 native, Komil fix)
model = None
try:
    print('  Trying trust_remote_code=False (Komil fix for KV cache)...')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, dtype=torch.bfloat16, device_map='auto',
        trust_remote_code=False, token=hf_token,
        attn_implementation='flash_attention_2',
    )
    print('  [OK] Native transformers Nemotron-H + FlashAttn2')
except Exception as e1:
    print(f'  [WARN] Native failed: {str(e1)[:100]}')
    try:
        print('  Trying trust_remote_code=True + FlashAttn2...')
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, dtype=torch.bfloat16, device_map='auto',
            trust_remote_code=True, token=hf_token,
            attn_implementation='flash_attention_2',
        )
        print('  [OK] trust_remote_code=True + FlashAttn2')
    except Exception as e2:
        print(f'  [WARN] FlashAttn2 failed: {str(e2)[:100]}')
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME, dtype=torch.bfloat16, device_map='auto',
            trust_remote_code=True, token=hf_token, attn_implementation='eager',
        )
        print('  [FALLBACK] eager attention')

model.config.use_cache = (FASE in ('1_diagnostic', '3_paired_validate_submit'))
print(f'  Model loaded {time.time()-t0:.0f}s | VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

is_trainable = (FASE == '2_train_ultraconservative')
model = PeftModel.from_pretrained(model, adapter_dir, adapter_name='default',
                                   token=hf_token, is_trainable=is_trainable)
print(f'  Adapter loaded (trainable={is_trainable})')

# ============================================================
# CORRECAO #3: SPEED TEST
# ============================================================
def speed_test():
    """Gera 1 sample teste, retorna tok/s."""
    print('\n[SPEED TEST] Gerando 1 sample teste...')
    model.eval()
    prompt_test = "What is 2+2? Please put your final answer inside `\\boxed{}`."
    msgs = [{'role': 'user', 'content': prompt_test}]
    inputs = tokenizer.apply_chat_template(msgs, return_tensors='pt',
                                            add_generation_prompt=True, enable_thinking=True).to('cuda')
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(inputs, max_new_tokens=200, temperature=0.0, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    elapsed = time.time() - t0
    n_tokens = out.shape[1] - inputs.shape[1]
    rate = n_tokens / elapsed if elapsed > 0 else 0
    print(f'  Generated {n_tokens} tokens in {elapsed:.1f}s = {rate:.1f} tok/s')
    if rate < SPEED_TEST_MIN_TOKS:
        print(f'  ❌ ABORT: {rate:.1f} tok/s < {SPEED_TEST_MIN_TOKS} tok/s threshold')
        print(f'  Komil bug detectado? Tente trust_remote_code=False ou transformers>=5.3.0')
        raise RuntimeError(f'Speed too low: {rate:.1f} tok/s')
    print(f'  ✅ Speed OK: {rate:.1f} tok/s >= {SPEED_TEST_MIN_TOKS} tok/s')
    return rate

# ============================================================
# Eval helper (com FIELD ROBUSTO + breakdown por dataset)
# ============================================================
KAGGLE_REGEX = re.compile(r'\\boxed\{([^}]*)(?:\}|$)')

def kaggle_verify(stored, predicted):
    stored = stored.strip(); predicted = predicted.strip()
    if re.fullmatch(r'[01]+', stored):
        return predicted.lower() == stored.lower()
    try:
        return math.isclose(float(stored), float(predicted), rel_tol=1e-2, abs_tol=1e-5)
    except:
        return predicted.lower() == stored.lower()

PROMPT_SUFFIX = '\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`'

def eval_samples(samples, label='', max_samples=None):
    """Eval com FIELD ROBUSTO + breakdown por _src."""
    model.eval()
    results = {
        'total': 0, 'correct': 0,
        'by_src': {'crypt': {'t': 0, 'c': 0}, 'eq': {'t': 0, 'c': 0}, 'synth': {'t': 0, 'c': 0}},
        'tipo_a': 0, 'tipo_b': 0, 'tipo_c': 0,
        'examples_c': [],
    }
    n = len(samples) if max_samples is None else min(len(samples), max_samples)
    t0 = time.time()
    for i, row in enumerate(samples[:n]):
        if i % 20 == 0 and i > 0:
            elapsed = time.time() - t0
            rate = i / elapsed if elapsed > 0 else 0
            eta = (n - i) / rate if rate > 0 else 0
            print(f'  [{label} {i}/{n}] {elapsed:.0f}s rate={rate:.2f}/s ETA={eta:.0f}s')

        prompt = (row.get('prompt') or '').strip()
        gold = get_gold(row)
        src = row.get('_src', 'unk')
        if not prompt or not gold:
            continue

        msgs = [{'role': 'user', 'content': prompt + PROMPT_SUFFIX}]
        inputs = tokenizer.apply_chat_template(msgs, return_tensors='pt',
                                                add_generation_prompt=True, enable_thinking=True).to('cuda')
        with torch.no_grad():
            out = model.generate(inputs, max_new_tokens=2048, temperature=0.0, do_sample=False,
                                  pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

        results['total'] += 1
        if src in results['by_src']:
            results['by_src'][src]['t'] += 1

        ms = KAGGLE_REGEX.findall(text)
        if not ms:
            results['tipo_a'] += 1
            continue
        extracted = next((m.strip() for m in reversed(ms) if m.strip()), None)
        if not extracted:
            results['tipo_a'] += 1
            continue

        if kaggle_verify(gold, extracted):
            results['correct'] += 1
            if src in results['by_src']:
                results['by_src'][src]['c'] += 1
            continue

        # Tipo B vs C
        cleanups = [
            extracted.replace('=', ' ').strip().split()[-1] if extracted.split() else '',
            re.sub(r'\s+', '', extracted),
            re.sub(r'[a-zA-Z/\s]', '', extracted),
            extracted.rstrip('.,;'),
        ]
        is_c = False
        for a in cleanups:
            if a and a != extracted and kaggle_verify(gold, a):
                is_c = True
                if len(results['examples_c']) < 5:
                    results['examples_c'].append({'gold': gold[:40], 'ext': extracted[:60], 'fix': a[:40]})
                break
        if is_c:
            results['tipo_c'] += 1
        else:
            results['tipo_b'] += 1
    return results

# ============================================================
# FASE 1: diagnostic_local (com HOLDOUT split)
# ============================================================
if FASE == '1_diagnostic':
    print('\n[FASE 1] Diagnostic em HOLDOUT eval (220 samples)')

    # Speed test
    rate = speed_test()

    # Load HOLDOUT eval data
    DATA_DIR = '/content/data'
    os.makedirs(DATA_DIR, exist_ok=True)
    train_data, eval_data, _, _ = load_datasets_with_split(DATA_DIR)
    print(f'\n  Train data: {len(train_data)} samples')
    print(f'  Eval HOLDOUT: {len(eval_data)} samples')

    # Eval
    print(f'\n  Eval V120 on HOLDOUT (~{len(eval_data)} samples)...')
    results = eval_samples(eval_data, label='V120')

    print(f'\n=== FASE 1 RESULTS (HOLDOUT) ===')
    tot = max(1, results['total'])
    print(f'Total eval: {results["total"]}')
    print(f'Kaggle parser score: {results["correct"]}/{results["total"]} = {100*results["correct"]/tot:.1f}%')
    print(f'\nBreakdown by dataset:')
    for src, d in results['by_src'].items():
        if d['t'] > 0:
            print(f'  {src}: {d["c"]}/{d["t"]} = {100*d["c"]/d["t"]:.1f}%')
    print(f'\nError tipos:')
    print(f'  Tipo A (no boxed):    {results["tipo_a"]:3} ({100*results["tipo_a"]/tot:.1f}%)')
    print(f'  Tipo B (math wrong):  {results["tipo_b"]:3} ({100*results["tipo_b"]/tot:.1f}%)')
    print(f'  Tipo C (format off):  {results["tipo_c"]:3} ({100*results["tipo_c"]/tot:.1f}%)')
    if results['examples_c']:
        print(f'\nExamples Tipo C:')
        for ex in results['examples_c']:
            print(f'  gold="{ex["gold"]}" ext="{ex["ext"]}" fix="{ex["fix"]}"')

    print(f'\n=== DECISAO ===')
    pct_c = 100 * results['tipo_c'] / tot
    pct_b = 100 * results['tipo_b'] / tot
    pct_a = 100 * results['tipo_a'] / tot
    if pct_c >= 5:
        print(f'>>> PARSER MISS CONFIRMADO ({pct_c:.1f}% Tipo C)')
        print(f'>>> NEXT: FASE 2 com FORMAT focus')
    elif pct_b > pct_c * 2:
        print(f'>>> Erros MATH ({pct_b:.1f}%)')
        print(f'>>> NEXT: FASE 2 com HEM math focus')
    elif pct_a >= 20:
        print(f'>>> Truncation issue ({pct_a:.1f}%)')
    else:
        print(f'>>> Resultado balanceado - FASE 2 standard')

    with open('/content/v158_diagnostic.json', 'w') as f:
        json.dump(results, f, indent=2)

    print('\n=== FASE 1 DONE ===')
    raise SystemExit(0)

# ============================================================
# FASE 2: train_ultraconservative (com TRAIN data, NAO eval)
# ============================================================
if FASE == '2_train_ultraconservative':
    print('\n[FASE 2] Train ULTRA-CONSERVATIVE (lr=1e-5, 80 steps)')

    model.enable_input_require_grads()
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant': True})
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    print(f'  Trainable: {trainable/1e6:.1f}M / {total/1e9:.2f}B')

    # Load TRAIN data (NAO eval)
    DATA_DIR = '/content/data'
    os.makedirs(DATA_DIR, exist_ok=True)
    train_data, eval_data, _, _ = load_datasets_with_split(DATA_DIR)
    print(f'\n  Train data (HOLDOUT excluded): {len(train_data)} samples')
    print(f'  Eval HOLDOUT (NAO usado em train): {len(eval_data)} samples')

    from datasets import Dataset
    all_data = []
    for row in train_data:
        p = (row.get('prompt') or '').strip()
        c = (row.get('cot') or row.get('generated_cot') or '').strip()
        if p and c:
            all_data.append({'prompt': p + PROMPT_SUFFIX, 'completion': c, 'src': row.get('_src', 'unk')})

    ORDER = {'eq': 0, 'crypt': 1, 'synth': 2}
    all_data.sort(key=lambda x: ORDER.get(x['src'], 99))
    ds = Dataset.from_list(all_data)
    print(f'  Total formatted: {len(ds)}')

    # Tokenize com truncation preserve final
    MAX_LENGTH = 6144
    def fmt(ex):
        msgs = [{'role': 'user', 'content': ex['prompt']},
                {'role': 'assistant', 'content': ex['completion']}]
        full = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False, enable_thinking=True)
        prm = tokenizer.apply_chat_template([msgs[0]], tokenize=False, add_generation_prompt=True, enable_thinking=True)
        full_ids = tokenizer(full, add_special_tokens=False)['input_ids']
        prm_ids = tokenizer(prm, add_special_tokens=False)['input_ids']
        if len(full_ids) > MAX_LENGTH:
            prefix = min(len(prm_ids), MAX_LENGTH // 3)
            suffix = MAX_LENGTH - prefix
            full_ids = full_ids[:prefix] + full_ids[-suffix:]
        labels = list(full_ids)
        n_prompt = min(len(prm_ids), len(labels))
        for i in range(n_prompt):
            labels[i] = -100
        return {'input_ids': full_ids, 'attention_mask': [1]*len(full_ids), 'labels': labels}

    tokenized = ds.map(fmt, remove_columns=ds.column_names, num_proc=4, desc='Tokenizing')
    seq_lens = [len(tokenized[i]['input_ids']) for i in range(min(500, len(tokenized)))]
    print(f'  Lens: min={min(seq_lens)} median={statistics.median(seq_lens):.0f} max={max(seq_lens)}')

    PAD_ID = tokenizer.pad_token_id or tokenizer.eos_token_id
    def collator(features):
        max_len = max(len(f['input_ids']) for f in features)
        max_len = ((max_len + 7) // 8) * 8
        ids, masks, lbls = [], [], []
        for f in features:
            pad = max_len - len(f['input_ids'])
            ids.append(f['input_ids'] + [PAD_ID]*pad)
            masks.append(f['attention_mask'] + [0]*pad)
            lbls.append(f['labels'] + [-100]*pad)
        return {'input_ids': torch.tensor(ids), 'attention_mask': torch.tensor(masks), 'labels': torch.tensor(lbls)}

    from transformers import Trainer, TrainingArguments
    OUT = '/content/output_v158'
    os.makedirs(OUT, exist_ok=True)

    train_args = TrainingArguments(
        output_dir=OUT,
        max_steps=80, per_device_train_batch_size=1, gradient_accumulation_steps=16,
        learning_rate=1e-5, lr_scheduler_type='cosine', warmup_steps=8,
        adam_beta1=0.9, adam_beta2=0.95, adam_epsilon=1e-8,
        weight_decay=0.0, max_grad_norm=0.5, label_smoothing_factor=0.0,
        logging_steps=5, save_steps=20, save_total_limit=4,
        bf16=True, optim='adamw_torch_fused',
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': True},
        remove_unused_columns=False, report_to='none',
        dataloader_num_workers=0, seed=42,
        neftune_noise_alpha=5,
    )

    trainer = Trainer(model=model, args=train_args, train_dataset=tokenized, data_collator=collator)
    print(f'  Steps: 80 | Warmup: 8 | Batch eff: 16 | LR: 1e-5')

    t0 = time.time()
    trainer.train()
    train_min = (time.time() - t0) / 60
    print(f'\n[OK] Training: {train_min:.1f} min')

    ADAPTER_OUT = f'{OUT}/final_adapter'
    trainer.save_model(ADAPTER_OUT)
    tokenizer.save_pretrained(ADAPTER_OUT)
    print(f'  Saved: {ADAPTER_OUT}')

    try:
        GD = '/content/drive/My Drive/KG1_v158_trained_adapter'
        if os.path.exists(GD): shutil.rmtree(GD)
        shutil.copytree(ADAPTER_OUT, GD)
        print(f'  GDrive backup: {GD}')
    except Exception as e:
        print(f'  GDrive: {e}')

    print('\n=== FASE 2 DONE - Run FASE 3 ===')
    raise SystemExit(0)

# ============================================================
# FASE 3: paired_validate_submit (HOLDOUT eval + per-dataset gate)
# ============================================================
if FASE == '3_paired_validate_submit':
    print('\n[FASE 3] Paired validation HOLDOUT + Submit')

    TRAINED_PATH = '/content/output_v158/final_adapter'
    if not os.path.exists(TRAINED_PATH):
        gd = '/content/drive/My Drive/KG1_v158_trained_adapter'
        if os.path.exists(gd):
            shutil.copytree(gd, TRAINED_PATH)
        else:
            raise RuntimeError('Trained adapter not found. Run FASE 2 first.')

    model.load_adapter(TRAINED_PATH, adapter_name='trained')

    # Speed test
    speed_test()

    # Load HOLDOUT eval (mesma split de FASE 1, NAO contaminado)
    DATA_DIR = '/content/data'
    os.makedirs(DATA_DIR, exist_ok=True)
    train_data, eval_data, _, _ = load_datasets_with_split(DATA_DIR)
    print(f'\n  HOLDOUT eval: {len(eval_data)} samples')

    # Eval V120
    print(f'\n  [V120 baseline] Eval {len(eval_data)} HOLDOUT samples...')
    model.set_adapter('default')
    res_v120 = eval_samples(eval_data, label='V120')

    # Eval TRAINED
    print(f'\n  [TRAINED] Eval {len(eval_data)} HOLDOUT samples...')
    model.set_adapter('trained')
    res_trained = eval_samples(eval_data, label='TRAINED')

    # Compare
    print(f'\n=== PAIRED VALIDATION (HOLDOUT) ===')
    tot = max(1, res_v120['total'])
    print(f'V120 baseline: {res_v120["correct"]}/{res_v120["total"]} = {100*res_v120["correct"]/tot:.1f}%')
    print(f'TRAINED:       {res_trained["correct"]}/{res_trained["total"]} = {100*res_trained["correct"]/tot:.1f}%')

    delta_total = res_trained['correct'] - res_v120['correct']
    print(f'\nDELTA total: {delta_total:+d} ({100*delta_total/tot:+.1f}%)')

    print(f'\nBreakdown per dataset:')
    no_go = False
    no_go_reason = ''
    for src in ['crypt', 'eq', 'synth']:
        v = res_v120['by_src'][src]
        t = res_trained['by_src'][src]
        if v['t'] == 0:
            continue
        d = t['c'] - v['c']
        v_pct = 100 * v['c'] / v['t']
        t_pct = 100 * t['c'] / t['t']
        marker = ''
        if d < 0:
            marker = ' ❌ REGRESSION'
            no_go = True
            no_go_reason = f'regressao em {src} ({d})'
        elif d == 0:
            marker = ' ⚠️ tie'
        else:
            marker = ' ✅'
        print(f'  {src}: V120={v["c"]}/{v["t"]}={v_pct:.1f}% | TRAINED={t["c"]}/{t["t"]}={t_pct:.1f}% | delta={d:+d}{marker}')

    # === NO-GO GATE COM PER-DATASET CHECK ===
    print(f'\n=== NO-GO GATE ===')
    if no_go:
        print(f'  >>> ❌ ABORT: {no_go_reason}')
        print(f'  >>> NAO submetendo. KEEP V120 baseline.')
    elif delta_total <= 0:
        print(f'  >>> ⚠️ NO IMPROVEMENT (delta_total={delta_total})')
        print(f'  >>> Nao recomendado submit')
        no_go = True
    else:
        print(f'  >>> ✅ APPROVED: delta_total={delta_total:+d}, sem regressao por dataset')

    if not no_go and not DRY_RUN:
        SUBMIT_DIR = '/content/submit_v158_trained'
        os.makedirs(SUBMIT_DIR, exist_ok=True)
        for fname in ['adapter_config.json', 'adapter_model.safetensors']:
            src = os.path.join(TRAINED_PATH, fname)
            if os.path.exists(src):
                shutil.copy(src, SUBMIT_DIR)

        zip_path = '/content/v158_trained.zip'
        subprocess.run(['zip', '-j', zip_path, *glob.glob(f'{SUBMIT_DIR}/*')], check=True)

        msg = f'v158 train_ultraconservative warmstart V120 - lr=1e-5 80steps batch=16 - holdout delta={delta_total:+d}'
        r = subprocess.run(['kaggle', 'competitions', 'submit', '-c',
                          'nvidia-nemotron-model-reasoning-challenge', '-f', zip_path, '-m', msg],
                         capture_output=True, text=True, timeout=600)
        print(f'\n  Submit: {r.stdout[:200]}{r.stderr[:200]}')
        print(f'  >>> SLOTS USED: 2/5')
    else:
        if DRY_RUN:
            print(f'  >>> DRY_RUN: nao submetendo')
        else:
            print(f'  >>> NOT submitting (gate failed)')

    # Save report
    with open('/content/v158_paired_report.json', 'w') as f:
        json.dump({'v120': res_v120, 'trained': res_trained, 'delta_total': delta_total, 'no_go': no_go}, f, indent=2)

    print('\n=== FASE 3 DONE ===')
    raise SystemExit(0)

print('\n[ERROR] Unknown FASE: ' + str(FASE))
